In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# Resolve dataset path dynamically
base_dir = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
data_path = base_dir / "data.csv"
if not data_path.exists():
    data_path = Path("../data.csv")

df = pd.read_csv(data_path)
df = df.drop_duplicates()
df.head()

In [ ]:
# Define categorical features and their fixed order of categories
CATEGORICAL_CATEGORIES = {
    "mainroad": ["no", "yes"],
    "guestroom": ["no", "yes"],
    "basement": ["no", "yes"],
    "hotwaterheating": ["no", "yes"],
    "airconditioning": ["no", "yes"],
    "prefarea": ["no", "yes"],
    "furnishingstatus": ["furnished", "semi-furnished", "unfurnished"],
}

X = df.drop("price", axis=1).copy()
y = df["price"]

for col, categories in CATEGORICAL_CATEGORIES.items():
    if col in X.columns:
        X[col] = pd.Categorical(X[col], categories=categories)

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)
X_encoded.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:,.2f}")
print(f"Root Mean Squared Error: {rmse:,.2f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
models_dir = base_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / "house_price_model.pkl")
feature_columns = X_encoded.columns.tolist()
joblib.dump(feature_columns, models_dir / "feature_columns.pkl")
print("Model and feature columns saved successfully to:", models_dir)